# Step 02. Frenet 좌표계

> **출처**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)  
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Step 01에서 Spline으로 **부드러운 Reference 경로**를 생성했습니다.  
이제 차량 위치를 이 경로 기준으로 표현합니다.  
**Frenet 좌표계** `(s, d)`를 사용하면 차선 추종·변경·장애물 회피를  
`d` 오프셋 조절 하나로 표현할 수 있습니다.

## 학습 목표

1. Frenet 좌표계 `(s, d)` 개념 이해
2. Reference 경로의 s축과 법선 방향(d축) 시각화
3. Cartesian ↔ Frenet 변환 구현 및 역변환 검증
4. d 오프셋별 후보 경로 생성 (Step 04 준비)

In [ ]:
%matplotlib inline
import sys, os, math
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from utils.cubic_spline import calc_spline_course
from utils.frenet import FrenetConverter

print('라이브러리 로드 완료')


## 1. Frenet 좌표계란?

### 이론 배경

절대 좌표 `(x, y)` 대신 Reference 경로를 기준으로 상대 위치를 표현합니다.

| 변수 | 의미 | 용도 |
|------|------|------|
| `s` | 경로를 따른 종방향 거리 [m] | 어느 지점인지 |
| `d` | 경로로부터 횡방향 오프셋 [m] | 경로에서 얼마나 벗어났는지 |

`d = 0`: Reference 경로 위  
`d > 0`: 경로 좌측 (차선 변경 좌측)  
`d < 0`: 경로 우측 (차선 변경 우측)

**왜 Frenet을 쓰는가?**  
복잡한 곡선 경로에서 차선 유지 = `d ≈ 0` 유지,  
장애물 회피 = `d`를 양수/음수로 조절.  
이 단순한 표현이 Local Planner 설계를 크게 간소화합니다.

In [ ]:
%matplotlib inline
import sys, os, math
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from utils.cubic_spline import calc_spline_course
from utils.frenet import FrenetConverter

print('라이브러리 로드 완료')


## 2. Reference 경로 생성

### 이론 배경

Spline 경로가 Frenet 좌표계의 **s축**이 됩니다.  
각 s 위치에서 경로의 yaw, 곡률(κ)도 함께 계산됩니다.

In [ ]:
wx = [0.0, 10.0, 20.5, 35.0, 50.0, 60.0]
wy = [0.0, -6.0,  5.0,  6.5,  0.0, -3.0]

rx, ry, ryaw, rk, rs = calc_spline_course(wx, wy, ds=0.5)

print('Reference 경로 정보')
print(f'  총 경로 길이 : {rs[-1]:.2f} m')
print(f'  경로 점 개수 : {len(rx)}개  (ds=0.5m)')
print(f'  최대 곡률    : {max(abs(k) for k in rk):.4f} rad/m')

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(rx, ry, '-b', lw=2, label='Reference Path (s-axis)')
ax.plot(wx, wy, 'o', ms=8, color='orange', label='Waypoints', zorder=3)
for i in range(0, len(rx), 60):
    ax.annotate(f's={rs[i]:.0f}m', (rx[i], ry[i]),
                fontsize=7, ha='center', va='bottom', color='navy')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend()
ax.set_title('Reference Path with Arc Length s Labels')
plt.tight_layout()
plt.show()


## 3. Frenet 좌표 시각화

### 이론 배경

경로의 각 점에서 **법선 방향**이 d축입니다.

법선 벡터: $n = (-\sin(\theta), \cos(\theta))$  
($\theta$: 경로의 yaw 각도)

d 오프셋 위치:  
$(x_d, y_d) = (x_r - d \sin\theta,\; y_r + d \cos\theta)$

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(rx, ry, '-b', lw=3, label='Reference (s-axis)', zorder=2)
ax.plot(wx, wy, 'o', ms=6, color='orange', alpha=0.7, zorder=3)

for i in range(10, len(rx)-10, 60):
    # 법선 벡터: (-sin(yaw), cos(yaw))
    nx = -math.sin(ryaw[i])
    ny =  math.cos(ryaw[i])
    for d_off in [-3, -1.5, 1.5, 3]:
        ex = rx[i] + d_off * nx
        ey = ry[i] + d_off * ny
        color = 'green' if d_off > 0 else 'red'
        label_text = f'+d={d_off}m (left)' if d_off == 3 else (
                     f'-d={abs(d_off)}m (right)' if d_off == -3 else None)
        ax.annotate('', xy=(ex, ey), xytext=(rx[i], ry[i]),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.8))
        if label_text:
            ax.text(ex, ey, label_text, fontsize=7, color=color, ha='center')

# 범례용 dummy
ax.plot([], [], '->', color='green', label='+d: left of path')
ax.plot([], [], '->', color='red',   label='-d: right of path')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
ax.set_title('Frenet Frame: Normal Direction (d-axis) at Each s Position')
plt.tight_layout()
plt.show()
print('s: 경로 진행 방향 / d: 경로에서 수직으로 벗어난 거리')
print('  d > 0 = 경로 좌측 / d < 0 = 경로 우측')


## 4. Cartesian ↔ Frenet 변환

### 이론 배경

변환에 필요한 값 (Reference 경로의 가장 가까운 점에서):

```
Cartesian -> Frenet:
  s     = 가장 가까운 Reference 점의 호 길이
  d     = 차량까지의 법선 거리 (방향 포함)
  s_dot = v * cos(Δθ) / (1 - κ*d)
  d_dot = (1-κ*d) * tan(Δθ)
```

역변환(Frenet → Cartesian)으로 복원하면 오차가 수 mm 이내여야 정상입니다.

In [ ]:
def find_nearest_idx(car_x, car_y, rx, ry):
    dists = [math.hypot(car_x - x, car_y - y) for x, y in zip(rx, ry)]
    return int(np.argmin(dists))

test_cases = [
    dict(car_x=20.0, car_y=8.0,  car_v=10.0, car_a=0.0, car_theta=0.3,  car_kappa=0.01, label='Vehicle A (left)'),
    dict(car_x=35.0, car_y=3.0,  car_v=12.0, car_a=0.0, car_theta=0.1,  car_kappa=0.0,  label='Vehicle B (right)'),
    dict(car_x=10.0, car_y=-6.0, car_v=8.0,  car_a=0.0, car_theta=-0.1, car_kappa=0.0,  label='Vehicle C (on path)'),
]

print('Cartesian -> Frenet 변환 결과')
print(f'  {"Vehicle":<18}  {"x":>6}  {"y":>6}  {"s":>8}  {"d":>8}  해석')
print('  ' + '-' * 68)

results = []
for tc in test_cases:
    idx = find_nearest_idx(tc['car_x'], tc['car_y'], rx, ry)
    s_cond, d_cond = FrenetConverter.cartesian_to_frenet(
        rs[idx], rx[idx], ry[idx], ryaw[idx], rk[idx], 0.0,
        tc['car_x'], tc['car_y'], tc['car_v'], tc['car_a'],
        tc['car_theta'], tc['car_kappa'])
    results.append((tc, s_cond, d_cond, idx))
    interp = 'left' if d_cond[0]>0.1 else ('right' if d_cond[0]<-0.1 else 'on path')
    print(f"  {tc['label']:<18}  {tc['car_x']:>6.1f}  {tc['car_y']:>6.1f}"
          f"  {s_cond[0]:>8.2f}  {d_cond[0]:>8.3f}  {interp}")

print()
print('  d > 0: 경로 좌측, d < 0: 경로 우측, d ≈ 0: 경로 위')


In [ ]:
# 변환된 (s, d)를 경로 위에 시각화 — 각 차량을 가장 가까운 Reference 점으로 투영
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(rx, ry, '-b', lw=2.5, alpha=0.6, label='Reference Path (s-axis)')

colors_v = ['#e74c3c', '#27ae60', '#8e44ad']
for (tc, s_cond, d_cond, idx), col in zip(results, colors_v):
    cxv, cyv = tc['car_x'], tc['car_y']
    # 차량 위치
    ax.plot(cxv, cyv, 'o', color=col, ms=11, zorder=5)
    # 가장 가까운 Reference 점 (투영점)
    ax.plot(rx[idx], ry[idx], 's', color=col, ms=8, alpha=0.7, zorder=4)
    # d 오프셋을 잇는 선 (법선 방향)
    ax.plot([cxv, rx[idx]], [cyv, ry[idx]], '--', color=col, lw=1.5)
    ax.annotate(f"{tc['label']}\ns={s_cond[0]:.1f}, d={d_cond[0]:+.2f}",
                (cxv, cyv), fontsize=8, color=col,
                xytext=(8, 8), textcoords='offset points')

# 범례용
ax.plot([], [], 'ok', label='Vehicle (Cartesian)')
ax.plot([], [], 'sk', alpha=0.6, label='Projection on path')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=8)
ax.set_title('Cartesian -> Frenet: each vehicle projected onto reference (dashed = d offset)')
plt.tight_layout()
plt.show()
print('점선 길이 = |d| (횡방향 거리),  사각형 위치의 s = 종방향 거리')


In [ ]:
print('역변환 검증 (Frenet -> Cartesian)')
print(f'  {"Vehicle":<18}  {"원래 (x,y)":>14}  {"복원 (x,y)":>14}  {"오차":>8}')
print('  ' + '-' * 62)

for tc, s_cond, d_cond, idx in results:
    x2, y2, *_ = FrenetConverter.frenet_to_cartesian(
        rs[idx], rx[idx], ry[idx], ryaw[idx], rk[idx], 0.0,
        s_cond, d_cond)
    err = math.hypot(tc['car_x'] - x2, tc['car_y'] - y2)
    print(f"  {tc['label']:<18}  ({tc['car_x']:5.1f},{tc['car_y']:5.1f})"
          f"  ({x2:6.3f},{y2:6.3f})  {err:>8.4f} m")
print()
print('오차 수 mm 이내 = 변환 정상  (오차는 nearest-point 근사에서 발생)')


## 5. d 오프셋별 후보 경로

### 이론 배경

d를 고정하면 Reference 경로와 평행한 후보 경로가 생성됩니다.  

$$x_{cand} = x_r - d \sin\theta_r, \quad y_{cand} = y_r + d \cos\theta_r$$

여러 d 값으로 후보 경로를 생성한 뒤 각 경로의 충돌 여부를 검사합니다.  
(Step 04에서 구체적으로 구현)

In [ ]:
d_offsets = np.arange(-3.0, 3.5, 1.0)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(rx, ry, '-b', lw=3, alpha=0.5, label='Reference Path')

colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(d_offsets)))
for d_off, color in zip(d_offsets, colors):
    cand_x = [rx[i] - math.sin(ryaw[i])*d_off for i in range(0, len(rx), 3)]
    cand_y = [ry[i] + math.cos(ryaw[i])*d_off for i in range(0, len(rx), 3)]
    ax.plot(cand_x, cand_y, '-', color=color, lw=1.5, alpha=0.8,
            label=f'd={d_off:+.1f}m')

ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=8, ncol=2)
ax.set_title('Candidate Paths for Different d Offsets')
plt.tight_layout()
plt.show()
print(f'후보 경로 수: {len(d_offsets)}개')
print('다음: Step 04에서 각 후보 경로의 장애물 충돌 검사')
